In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, rand, expr
import uuid

# Initialize Spark session in Fabric
spark = SparkSession.builder.appName("DiasporaFintechLogs").getOrCreate()

# Create realistic base data (simulating a diaspora remittance app)
# Features: US source states to various home countries
data = [(i, 
         f"USER_{1000 + (i % 100)}", 
         float((i % 50) * 25 + 10),  # Base amounts
         "USD", 
         "US-NY" if i % 2 == 0 else "US-CA", 
         "NGN" if i % 3 == 0 else "INR" if i % i%5==0 else "MXN") 
        for i in range(1, 5001)]

columns = ["log_id", "user_id", "amount_usd", "source_currency", "source_location", "target_currency"]
raw_df = spark.createDataFrame(data, columns)

# Inject synthetic real-world anomalies & fraud indicators
# Process raw records and apply type-safe date arithmetic
processed_logs_df = raw_df.withColumn(
    # FIX: Multiplies the numeric scalar directly against an explicit interval literal
    "timestamp", expr("current_timestamp() - (rand() * interval 1000 minute)")
).withColumn(
    "device_ip", when(rand() > 0.95, "192.168.99.99").otherwise("10.0.0.1")
).withColumn(
    "is_fraud", 
    when((col("amount_usd") > 1100) & (col("source_location") == "US-NY"), 1)
    .when((col("device_ip") == "192.168.99.99") & (col("target_currency") == "NGN"), 1)
    .otherwise(0)
)

# Save as Bronze Layer Table (Append-only Raw Storage)
processed_logs_df.write.format("delta").mode("overwrite").saveAsTable("bronze_remittance_logs")
print("Bronze layer created successfully!")


StatementMeta(, a95a726a-272f-43b0-8b4f-e588a0edbfc7, 3, Finished, Available, Finished, False)

Bronze layer created successfully!


In [3]:
# # Silver code 

# from pyspark.sql.functions import col, when

# # 1. Read your freshly created data directly from the Bronze Delta table
# bronze_df = spark.read.table("bronze_remittance_logs")

# # 2. Execute Data Cleansing & Financial Feature Engineering
# # We create an explicit risk indicator flag for high-value diaspora transfers ($500+)
# # and select only the structured columns our machine learning model requires.
# silver_features_df = bronze_df.withColumn(
#     "is_high_value", when(col("amount_usd") > 500, 1).otherwise(0)
# ).select(
#     "amount_usd", 
#     "is_high_value",
#     "source_location", 
#     "target_currency", 
#     "is_fraud"
# )

# # 3. Write out to the Silver Layer Table
# silver_features_df.write.format("delta").mode("overwrite").saveAsTable("silver_fraud_features")

# print("Silver layer features engineered and saved successfully!")
# display(silver_features_df.limit(5))

# from pyspark.sql.functions import col, when

# bronze_df = spark.read.table("bronze_remittance_logs")

# # FIX: Explicitly cast 'is_high_value' to a double and remove system metadata tags
# silver_features_df = bronze_df.withColumn(
#     "is_high_value", when(col("amount_usd") > 500, 1.0).otherwise(0.0).cast("double")
# ).select(
#     "amount_usd", 
#     "is_high_value",
#     "source_location", 
#     "target_currency", 
#     "is_fraud"
# )

# # Overwrite old silver layer
# silver_features_df.write.format("delta").mode("overwrite").saveAsTable("silver_fraud_features")
# print("Silver layer cleaned with explicit data types!")

from pyspark.sql.functions import col, when

bronze_df = spark.read.table("bronze_remittance_logs")

# Explicitly cast 'is_high_value' to a double
silver_features_df = bronze_df.withColumn(
    "is_high_value", when(col("amount_usd") > 500, 1.0).otherwise(0.0).cast("double")
).select(
    "amount_usd", 
    "is_high_value",
    "source_location", 
    "target_currency", 
    "is_fraud"
)

# FIX: Add option("overwriteSchema", "true") to drop the old integer structure
silver_features_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_fraud_features")

print("Silver layer schema overwritten and saved successfully!")




StatementMeta(, 6b4b2d82-1f4a-4644-8470-035d165d7a00, 5, Finished, Available, Finished, False)

Silver layer schema overwritten and saved successfully!


In [10]:
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from synapse.ml.lightgbm import LightGBMClassifier

# 1. Clear session memory to dump lingering metadata caches
spark.catalog.clearCache()

# 2. Extract a completely fresh slice of data from Silver storage
base_silver_df = spark.read.table("silver_fraud_features")

# 3. Transform textual identifiers into explicit numeric sequences
indexer_loc = StringIndexer(inputCol="source_location", outputCol="idx_loc", handleInvalid="keep")
indexer_curr = StringIndexer(inputCol="target_currency", outputCol="idx_curr", handleInvalid="keep")

# Apply indexers directly to build a clean dataset free of default column attributes
indexed_df = indexer_loc.fit(base_silver_df).transform(base_silver_df)
final_df = indexer_curr.fit(indexed_df).transform(indexed_df)

# 4. Partition your clean data into Train (80%) and Test (20%) datasets
train_data, test_data = final_df.randomSplit([0.8, 0.2], seed=42)

# 5. FIX: Pass explicit column strings directly to slotNames inside LightGBMClassifier
# This bypasses VectorAssembler and prevents the duplicate 'Column_' metadata crash
lgbm = LightGBMClassifier(
    labelCol="is_fraud",
    objective="binary",
    numLeaves=31,
    learningRate=0.1,
    featuresCol="amount_usd",                    # Native primary float column index
    categoricalSlotNames=["idx_loc", "idx_curr"] # Direct mapping arrays for LightGBM
)

# 6. Fit the model directly onto training data
model = lgbm.fit(train_data)

# 7. Compute predictions and save to production Gold Storage
predictions = model.transform(test_data)
predictions.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fraud_predictions")

print("Gold Layer trained successfully! System bypass complete.")
display(predictions.select("amount_usd", "source_location", "target_currency", "is_fraud", "prediction").limit(5))


StatementMeta(, 6b4b2d82-1f4a-4644-8470-035d165d7a00, 12, Finished, Available, Finished, False)

IllegalArgumentException: requirement failed: Column amount_usd must be of type class org.apache.spark.ml.linalg.VectorUDT:struct<type:tinyint,size:int,indices:array<int>,values:array<double>> but was actually class org.apache.spark.sql.types.DoubleType$:double.